In [ ]:
import os, json, time, queue, random, threading, warnings
from pathlib import Path
import numpy as np, rasterio, torch, torch.nn.functional as F
from scipy import ndimage
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.benchmark = True
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train" / "bs").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("Сначала выполните prepare_data.py --scope train")
TRAIN = PROJECT / "data" / "train" / "bs"
OUT = PROJECT / "outputs" / "training" / "burn_severity" / "notebook_weights"
OUT.mkdir(exist_ok=True)
DEVICE = "cuda"
SIZE, CROP, CROPS_PER_CHIP, BATCH = 512, 256, 4, 8
EPOCH_SCHEDULE, EPOCH_STOP, LR, WEIGHT_DECAY = 60, 25, 1e-3, 1e-4
CKPT_EVERY = 5
BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8A", "B11", "B12"]
LC_GROUP = {10: "forest", 20: "forest", 30: "grass", 40: "crop", 90: "wetland"}
DNBR_THRESH = {
    "forest": (0.1, 0.27, 0.66),
    "grass": (0.062, 0.204, 0.386),
    "crop": (0.07, 0.177, 0.38),
    "wetland": (0.075, 0.331, 0.677),
}
print(torch.__version__, "|", torch.cuda.get_device_name(0), "| seed", SEED)

In [ ]:
def read(folder, cid, suffix):
    if folder == "aux" and not (TRAIN / folder).is_dir():
        folder = "aux_data"
    with rasterio.open(TRAIN / folder / f"{cid}_{suffix}.tif") as s:
        return {n: s.read(i).astype(np.float32) for i, n in enumerate(s.descriptions, 1)}


def nd(a, b):
    return (a - b) / np.maximum(a + b, 1.0)


def chip_tensor(cid):
    pre, post = read("sentinel2_pre", cid, "Sentinel-2_pre"), read("sentinel2_post", cid, "Sentinel-2_post")
    s1a, s1b = read("sentinel1_pre", cid, "Sentinel-1_pre"), read("sentinel1_post", cid, "Sentinel-1_post")
    aux = read("aux", cid, "AUX")
    sm = lambda x: ndimage.uniform_filter(x, 3)
    nbr_pre, nbr_post = nd(sm(pre["B8A"]), sm(pre["B12"])), nd(sm(post["B8A"]), sm(post["B12"]))
    dnbr = nbr_pre - nbr_post
    lc = aux["landcover"].astype(np.int32)
    t = np.zeros((3,) + lc.shape, np.float32)
    for i in range(3):
        t[i] = DNBR_THRESH["grass"][i]
    for code, group in LC_GROUP.items():
        for i in range(3):
            t[i][lc == code] = DNBR_THRESH[group][i]
    sev = np.where(dnbr >= t[2], 3, np.where(dnbr >= t[1], 2, np.where(dnbr >= t[0], 1, 0)))
    r = lambda d, b: d[b] / 10000.0
    x = [r(pre, b) for b in BANDS] + [r(post, b) for b in BANDS]
    x += [
        dnbr,
        nd(pre["B8A"], pre["B4"]) - nd(post["B8A"], post["B4"]),
        nd(pre["B11"], pre["B12"]) - nd(post["B11"], post["B12"]),
        nbr_pre,
        nbr_post,
        (10 * r(pre, "B12") - 9.8 * r(pre, "B11")) - (10 * r(post, "B12") - 9.8 * r(post, "B11")),
        dnbr - t[0],
        sev / 3.0,
        s1a["VV"] / 1000,
        s1a["VH"] / 1000,
        s1b["VV"] / 1000,
        s1b["VH"] / 1000,
        (s1b["VV"] - s1a["VV"]) / 1000,
        (s1b["VH"] - s1a["VH"]) / 1000,
        aux["dem"] / 500.0,
        aux["slope"] / 30.0,
        np.isin(lc, [10, 20]),
        lc == 30,
        lc == 40,
        lc == 90,
        ~np.isin(lc, [10, 20, 30, 40, 90]),
        np.isin(post["SCL"], [3, 8, 9, 10]),
    ]
    return np.nan_to_num(np.stack([np.asarray(a, np.float32) for a in x]), nan=0, posinf=0, neginf=0).astype(
        np.float16
    )


CHIPS = sorted(p.name.split("_MASK")[0] for p in (TRAIN / "masks").glob("*_MASK.tif"))
XA = np.zeros((len(CHIPS), 40, SIZE, SIZE), np.float16)
BURN = np.zeros((len(CHIPS), SIZE, SIZE), np.float32)
t0 = time.time()
for i, cid in enumerate(tqdm(CHIPS, desc="тензоры")):
    XA[i] = chip_tensor(cid)
    with rasterio.open(TRAIN / "masks" / f"{cid}_MASK.tif") as s:
        m = s.read(1)
    BURN[i] = (m >= 1) & (m <= 3)
print(
    f"чипов {len(CHIPS)} | тензор {XA.shape}, {XA.nbytes / 1e9:.1f} ГБ | гари {100 * BURN.mean():.2f}% пикселей | {time.time() - t0:.0f}с"
)

In [ ]:
xs = XA[:, :, ::8, ::8].astype(np.float32)
MEAN_NP, STD_NP = xs.mean((0, 2, 3)), xs.std((0, 2, 3)) + 1e-6
json.dump({"mean": MEAN_NP.tolist(), "std": STD_NP.tolist()}, open(OUT / "norm.json", "w"))
MEAN = torch.tensor(MEAN_NP).view(1, -1, 1, 1).to(DEVICE)
STD = torch.tensor(STD_NP).view(1, -1, 1, 1).to(DEVICE)
print("нормировка по 40 каналам сохранена:", OUT / "norm.json")

In [ ]:
def producer(rng, q):
    items = [
        (c, rng.randint(0, SIZE - CROP + 1), rng.randint(0, SIZE - CROP + 1))
        for c in range(len(CHIPS))
        for _ in range(CROPS_PER_CHIP)
    ]
    rng.shuffle(items)
    for i in range(0, len(items), BATCH):
        xs, ys = [], []
        for c, r, s in items[i : i + BATCH]:
            x, y = (
                XA[c, :, r : r + CROP, s : s + CROP].astype(np.float32),
                BURN[c, r : r + CROP, s : s + CROP],
            )
            k = rng.randint(4)
            x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
            if rng.rand() < 0.5:
                x, y = x[:, :, ::-1], y[:, ::-1]
            xs.append(np.ascontiguousarray(x))
            ys.append(np.ascontiguousarray(y))
        q.put(
            (
                torch.from_numpy(np.stack(xs)).pin_memory(),
                torch.from_numpy(np.stack(ys))[:, None].pin_memory(),
            )
        )
    q.put(None)


def batches(rng):
    q = queue.Queue(maxsize=8)
    threading.Thread(target=producer, args=(rng, q), daemon=True).start()
    while (item := q.get()) is not None:
        yield item


def loss_fn(logits, y):
    p = torch.sigmoid(logits)
    dice = 1 - (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)
    return F.binary_cross_entropy_with_logits(logits, y) + dice


STEPS_PER_EPOCH = (len(CHIPS) * CROPS_PER_CHIP + BATCH - 1) // BATCH
print(f"шагов в эпохе {STEPS_PER_EPOCH} | расписание на {EPOCH_SCHEDULE} эпох, остановка на {EPOCH_STOP}")

In [ ]:
model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=XA.shape[1], classes=1).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=LR, total_steps=EPOCH_SCHEDULE * STEPS_PER_EPOCH, pct_start=0.1
)
scaler = torch.amp.GradScaler()
rng = np.random.RandomState(SEED)
history = []
t0 = time.time()
for ep in tqdm(range(1, EPOCH_STOP + 1), desc="эпохи"):
    model.train()
    tot = n = 0
    for x, y in tqdm(batches(rng), total=STEPS_PER_EPOCH, desc=f"эпоха {ep}", leave=False):
        x = (x.to(DEVICE, non_blocking=True) - MEAN) / STD
        y = y.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(x).float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        tot += loss.item() * len(x)
        n += len(x)
    history.append({"epoch": ep, "loss": tot / n, "lr": sched.get_last_lr()[0], "sec": time.time() - t0})
    tqdm.write(
        f"эпоха {ep}/{EPOCH_STOP}: loss {tot / n:.4f} | lr {sched.get_last_lr()[0]:.2e} | {time.time() - t0:.0f}с"
    )
    if ep % CKPT_EVERY == 0 or ep == EPOCH_STOP:
        torch.save(model.state_dict(), OUT / f"unet_ep{ep}.pt")

In [ ]:
torch.save(model.state_dict(), OUT / "unet_final.pt")
json.dump(
    {
        "architecture": "smp.Unet(resnet34)",
        "in_channels": int(XA.shape[1]),
        "epoch_schedule": EPOCH_SCHEDULE,
        "epoch_stop": EPOCH_STOP,
        "batch": BATCH,
        "crop": CROP,
        "lr": LR,
        "seed": SEED,
        "history": history,
    },
    open(OUT / "train_config.json", "w"),
    indent=1,
)
for p in sorted(OUT.iterdir()):
    print(f"{p.name:<22} {p.stat().st_size / 1e6:7.1f} МБ")